# 01 - Exploratory data analysis

Image sizes, GSD, class pixel distribution and sample image/mask pairs for OpenEarthMap and LoveDA. Run `scripts/download_data.py` first. Sections skip themselves if a dataset is missing.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from PIL import Image

RAW = Path("../data/raw")
SAMPLE = 200  # images per dataset used for statistics
random.seed(0)

# Class names (index = raw mask value). LoveDA encoding is verified below.
OEM_CLASSES = [
    "unlabeled",
    "bareland",
    "rangeland",
    "developed",
    "road",
    "tree",
    "water",
    "agriculture",
    "building",
]
LOVEDA_CLASSES = [
    "no-data",
    "background",
    "building",
    "road",
    "water",
    "barren",
    "forest",
    "agriculture",
]

## File discovery

In [ ]:
def find_pairs_oem():
    """OpenEarthMap: <region>/images/*.tif with matching <region>/labels/*.tif."""
    pairs = []
    for img in (RAW / "openearthmap").rglob("images/*.tif"):
        lab = img.parent.parent / "labels" / img.name
        if lab.exists():
            pairs.append((img, lab))
    return sorted(pairs)


def find_pairs_loveda(splits=("Train", "Val")):
    """LoveDA: <Split>/<Scene>/images_png/*.png with matching masks_png/*.png."""
    pairs = []
    for split in splits:
        for img in (RAW / "loveda" / split).rglob("images_png/*.png"):
            lab = img.parent.parent / "masks_png" / img.name
            if lab.exists():
                pairs.append((img, lab))
    return sorted(pairs)


oem_pairs, loveda_pairs = find_pairs_oem(), find_pairs_loveda()
print(f"OpenEarthMap pairs: {len(oem_pairs)}\nLoveDA pairs:       {len(loveda_pairs)}")

## Image size and GSD

OpenEarthMap GSD comes from the GeoTIFF transform; LoveDA PNGs have no georeference so the published 0.3 m is used.

In [ ]:
def oem_meta(img_path):
    with rasterio.open(img_path) as src:
        gsd = abs(src.transform.a)
        # Tiles may be geographic (degrees); convert to metres at the tile latitude.
        if src.crs is not None and src.crs.is_geographic:
            lat = (src.bounds.top + src.bounds.bottom) / 2
            gsd = gsd * 111_320 * np.cos(np.radians(lat))
        return dict(width=src.width, height=src.height, bands=src.count, gsd_m=round(gsd, 3))


def loveda_meta(img_path):
    with Image.open(img_path) as im:
        return dict(width=im.width, height=im.height, bands=len(im.getbands()), gsd_m=0.3)


rows = []
for name, pairs, fn in [
    ("OpenEarthMap", oem_pairs, oem_meta),
    ("LoveDA", loveda_pairs, loveda_meta),
]:
    for img, _ in random.sample(pairs, min(SAMPLE, len(pairs))) if pairs else []:
        rows.append(dict(dataset=name, **fn(img)))
meta = pd.DataFrame(rows)
if len(meta):
    display(
        meta.groupby("dataset").agg(
            n=("width", "size"),
            width=("width", "unique"),
            height=("height", "unique"),
            gsd_min=("gsd_m", "min"),
            gsd_median=("gsd_m", "median"),
            gsd_max=("gsd_m", "max"),
        )
    )

## Class pixel distribution

In [ ]:
def read_mask(path):
    if path.suffix == ".tif":
        with rasterio.open(path) as src:
            return src.read(1)
    return np.array(Image.open(path))


def class_counts(pairs, n_classes):
    counts = np.zeros(n_classes, dtype=np.int64)
    for _, lab in pairs:
        m = read_mask(lab)
        counts += np.bincount(m.ravel(), minlength=n_classes)[:n_classes]
    return counts


dist = {}
if oem_pairs:
    dist["OpenEarthMap"] = pd.Series(class_counts(oem_pairs, 9), index=OEM_CLASSES)
if loveda_pairs:
    dist["LoveDA"] = pd.Series(class_counts(loveda_pairs, 8), index=LOVEDA_CLASSES)

fig, axes = plt.subplots(1, max(len(dist), 1), figsize=(6 * max(len(dist), 1), 4), squeeze=False)
for ax, (name, s) in zip(axes[0], dist.items(), strict=False):
    (s / s.sum() * 100).plot.barh(ax=ax, title=f"{name}: % of pixels")
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../docs/figures/eda_class_distribution.png", dpi=120)
for name, s in dist.items():
    print(name)
    print((s / s.sum() * 100).round(2).to_string(), "\n")

### LoveDA mask encoding check

Sources disagree on whether value 0 is no-data or background. If value 0 is very rare and value 1 is common, the encoding is 0 = no-data, 1 = background (as assumed above).

In [ ]:
if loveda_pairs:
    vals = np.unique(
        np.concatenate(
            [
                np.unique(read_mask(lab_path))
                for _, lab_path in random.sample(loveda_pairs, min(50, len(loveda_pairs)))
            ]
        )
    )
    print("Distinct mask values:", vals)
    print((dist["LoveDA"] / dist["LoveDA"].sum() * 100).round(3)[["no-data", "background"]])

## Sample image / mask pairs

In [ ]:
OEM_COLORS = np.array(
    [
        [0, 0, 0],
        [188, 143, 143],
        [144, 238, 144],
        [210, 180, 140],
        [128, 128, 128],
        [34, 139, 34],
        [30, 144, 255],
        [154, 205, 50],
        [220, 50, 47],
    ],
    dtype=np.uint8,
)
LOVEDA_COLORS = np.array(
    [
        [0, 0, 0],
        [255, 255, 255],
        [220, 50, 47],
        [128, 128, 128],
        [30, 144, 255],
        [188, 143, 143],
        [34, 139, 34],
        [154, 205, 50],
    ],
    dtype=np.uint8,
)


def show_samples(title, pairs, colors, n=4):
    if not pairs:
        print(f"{title}: no data found")
        return
    fig, axes = plt.subplots(2, n, figsize=(3.5 * n, 7))
    for i, (img, lab) in enumerate(random.sample(pairs, n)):
        if img.suffix == ".tif":
            with rasterio.open(img) as src:
                rgb = np.moveaxis(src.read([1, 2, 3]), 0, -1)
        else:
            rgb = np.array(Image.open(img).convert("RGB"))
        axes[0, i].imshow(rgb)
        axes[1, i].imshow(colors[read_mask(lab)])
        axes[0, i].set_title(img.stem[:22], fontsize=8)
        for a in axes[:, i]:
            a.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(f"../docs/figures/eda_samples_{title.lower()}.png", dpi=80)


show_samples("OpenEarthMap", oem_pairs, OEM_COLORS)
show_samples("LoveDA", loveda_pairs, LOVEDA_COLORS)

## Observations

Fill in after running on the real data: dominant classes, rare classes, GSD spread, visible label noise. These feed `docs/reports/week-01.md`.